<a href="https://colab.research.google.com/github/TU-USUARIO/labo1-colabs/blob/main/04_Propagacion_de_incertezas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Colab 04 — Propagación de incertezas y errores sistemáticos**Laboratorio 1 · Clase 4****Objetivos.**1. Propagar incertezas a magnitudes calculadas, sin memorizar casos particulares.2. Identificar **cuál de tus mediciones domina** la incerteza final — que es la única razón   práctica para propagar.3. Entender por qué promediar no corrige un error sistemático, y verlo numéricamente.**Requisitos previos:** Colabs 01 a 03.> **Antes de empezar:** hacé `Archivo → Guardar una copia en Drive`. Vas a trabajar sobre *tu* copia; el original queda intacto para el resto del curso.

In [ ]:
import numpy as npimport sympy as spimport matplotlib.pyplot as pltnp.random.seed(20260902)

---## 1. La fórmula generalSi $f$ depende de magnitudes medidas $x_1, \\dots, x_n$ con incertezas $\\sigma_1, \\dots, \\sigma_n$**independientes**, entonces$$ \\sigma_f^2 = \\sum_{i=1}^{n} \\left(\\frac{\\partial f}{\\partial x_i}\\right)^2 \\sigma_i^2 $$Todos los casos que aparecen en los libros (suma, producto, potencia) salen de acá. No hace faltamemorizarlos; conviene entender esta expresión y saber derivar.Dos observaciones que suelen pasarse por alto:- Los términos se suman **en cuadratura**. Una contribución que sea un tercio de otra aporta un  noveno a la varianza: es decir, casi nada. Eso significa que en la práctica **una o dos mediciones  dominan el error total** y el resto no importa.- La fórmula supone independencia. Si dos magnitudes están correlacionadas (típico cuando salen del  mismo ajuste), hay un término cruzado adicional. Volvemos a esto en el Colab 06.

---## 2. Derivadas simbólicas con SymPyDerivar a mano una expresión con cinco variables es una fuente de errores gratuita. SymPy lo haceexacto.

In [ ]:
# Ejemplo: densidad de un cilindro,  rho = m / (pi * r^2 * h)m, r, h = sp.symbols('m r h', positive=True)rho = m / (sp.pi * r**2 * h)print("ρ =", rho)print()for var in (m, r, h):    print(f"∂ρ/∂{var} =", sp.simplify(sp.diff(rho, var)))

In [ ]:
def propagar(expr, variables, valores, errores, nombre='f'):    """Propaga incertezas de una expresión simbólica.    expr      : expresión de SymPy    variables : lista de símbolos    valores   : dict {símbolo: valor medido}    errores   : dict {símbolo: incerteza}    Devuelve (valor, incerteza) e imprime la tabla de contribuciones.    """    # varianza simbólica    var_expr = sum((sp.diff(expr, v))**2 * sp.Symbol(f'sigma_{v}')**2 for v in variables)    subs = dict(valores)    valor = float(expr.subs(subs))    contribuciones = {}    for v in variables:        d = float(sp.diff(expr, v).subs(subs))        contribuciones[v] = (d * errores[v])**2    var_total = sum(contribuciones.values())    sigma = np.sqrt(var_total)    print(f"{nombre} = {valor:.6g} ± {sigma:.3g}   ({100*sigma/abs(valor):.2f} % relativo)")    print("-" * 58)    print(f"{'variable':<10}{'contribución a σ²':>20}{'% del total':>16}")    print("-" * 58)    for v in sorted(variables, key=lambda v: -contribuciones[v]):        print(f"{str(v):<10}{contribuciones[v]:>20.4g}{100*contribuciones[v]/var_total:>15.1f}%")    print("-" * 58)    return valor, sigma

In [ ]:
# Un caso concreto: cilindro metálicovalores = {m: 0.0821, r: 0.00625, h: 0.0402}          # kg, m, merrores = {m: 0.0002,  r: 0.00002, h: 0.0001}         # kg, m, mvalor, sigma = propagar(rho, [m, r, h], valores, errores, nombre='ρ [kg/m³]')

Leé la tabla de contribuciones, que es el resultado útil: te dice **dónde invertir esfuerzoexperimental**. Si el radio aporta el 70 % de la varianza, medir la masa con una balanza mejor no vaa cambiar nada. Conseguir un calibre mejor, sí.Este razonamiento es exactamente el que se usa al diseñar un experimento de investigación, y es loque se le va a pedir en la Práctica Especial: estimar *a priori* qué error esperás y de dónde viene.> **Ejercicio 4.1.** Bajá a la mitad la incerteza de la variable dominante y volvé a correr la> celda. ¿Cuánto mejoró el resultado? Ahora bajá a la mitad la de la variable menos importante.> Compará.

---## 3. Del símbolo al número, en cantidad: `lambdify`Cuando tenés que propagar sobre muchos puntos (por ejemplo, una columna entera de datos), evaluarcon `subs()` es lentísimo. `lambdify` convierte la expresión simbólica en una función de NumPy.

In [ ]:
# expresión simbólica de la incerteza propagadasig_m, sig_r, sig_h = sp.symbols('sigma_m sigma_r sigma_h', positive=True)var_rho = ((sp.diff(rho, m))**2 * sig_m**2 +           (sp.diff(rho, r))**2 * sig_r**2 +           (sp.diff(rho, h))**2 * sig_h**2)sigma_rho = sp.sqrt(var_rho)f_rho   = sp.lambdify((m, r, h), rho, 'numpy')f_sigma = sp.lambdify((m, r, h, sig_m, sig_r, sig_h), sigma_rho, 'numpy')# ahora sobre arreglos completosmasas  = np.array([0.0821, 0.0819, 0.0824, 0.0820])radios = np.full(4, 0.00625)alturas = np.array([0.0402, 0.0401, 0.0403, 0.0402])rhos = f_rho(masas, radios, alturas)sigs = f_sigma(masas, radios, alturas, 0.0002, 0.00002, 0.0001)for i, (a, b) in enumerate(zip(rhos, sigs), 1):    print(f"muestra {i}: ρ = {a:8.1f} ± {b:5.1f} kg/m³")

---## 4. El punto importante: promediar no corrige un sistemáticoSimulamos dos instrumentos midiendo la misma magnitud, cuyo valor verdadero es 10,000:- **Instrumento A:** ruidoso pero sin sesgo ($\\sigma = 0{,}5$, sesgo 0).- **Instrumento B:** muy poco ruidoso pero descalibrado ($\\sigma = 0{,}05$, sesgo $+0{,}3$).Miramos qué pasa con el promedio de cada uno a medida que crece $N$.

In [ ]:
VERDADERO = 10.000N_max = 2000ns = np.arange(1, N_max + 1)A = np.random.normal(VERDADERO,       0.50, N_max)   # ruidoso, insesgadoB = np.random.normal(VERDADERO + 0.30, 0.05, N_max)  # preciso, sesgadomediaA = np.cumsum(A) / nsmediaB = np.cumsum(B) / nsfig, ax = plt.subplots(figsize=(7.5, 4.4))ax.axhline(VERDADERO, color='k', ls='--', lw=1.2, label='valor verdadero')ax.plot(ns, mediaA, lw=1.2, label='A: ruidoso, sin sesgo')ax.plot(ns, mediaB, lw=1.2, color='crimson', label='B: preciso, con sesgo +0,30')ax.set_xscale('log')ax.set_xlabel('Cantidad de mediciones promediadas $N$')ax.set_ylabel('Promedio acumulado')ax.set_title('Promediar reduce el error aleatorio; no toca el sistemático')ax.grid(alpha=0.3); ax.legend()fig.tight_layout(); plt.show()

In [ ]:
for n in [10, 100, 1000, 2000]:    xa, sa = A[:n].mean(), np.std(A[:n], ddof=1)/np.sqrt(n)    xb, sb = B[:n].mean(), np.std(B[:n], ddof=1)/np.sqrt(n)    za = abs(xa - VERDADERO)/sa    zb = abs(xb - VERDADERO)/sb    print(f"N = {n:5d} | A: {xa:.4f} ± {sa:.4f}  (z = {za:5.1f})"          f"   | B: {xb:.4f} ± {sb:.4f}  (z = {zb:6.1f})")

Mirá la columna de B. A medida que $N$ crece, el error de la media baja, el resultado se ve cada vez**más preciso**, y la discrepancia con el valor verdadero medida en unidades de $\\sigma$ crece sinparar. Con $N = 2000$, B reporta un resultado con cuatro cifras significativas que está a decenas de$\\sigma$ del valor correcto.Ése es el escenario que hay que aprender a temer: **un resultado que parece cada vez mejor y es cadavez más falso**. La única defensa es la calibración y la comparación con un método independiente,no el aumento de $N$.> **Caso real.** En medición de curvas I–V de junturas memristivas, la histéresis por dirección de> barrido produce exactamente este efecto: promediar la rama de ida con la de vuelta da un resultado> con dispersión chica y un valor que no corresponde a ningún estado físico del dispositivo. La> dispersión pequeña es, en ese caso, una señal de alarma y no de calidad.

---## 5. Ejercicios**4.2.** Propagá la incerteza de $g$ en la medición con péndulo, $g = 4\\pi^2 L / T^2$, con$L = (1{,}000 \\pm 0{,}002)$ m y $T = (2{,}006 \\pm 0{,}004)$ s. ¿Qué variable domina? ¿Cuál convienemedir mejor?**4.3.** Volvé a tus datos de tiempo de reacción del Colab 02. Si tu cronómetro tuviera un retardosistemático de +15 ms, ¿cómo cambiaría tu resultado? ¿Lo detectarías mirando la dispersión?¿Cómo lo detectarías?**4.4.** Escribí la propagación para $f = x/y$ y verificá simbólicamente con SymPy que se reduce a$(\\sigma_f/f)^2 = (\\sigma_x/x)^2 + (\\sigma_y/y)^2$. Es el caso que más vas a usar y conviene haberloderivado una vez.**4.5.** *(criterio)* Tenés dos mediciones de la misma longitud: $(10{,}0 \\pm 0{,}5)$ cm y$(10{,}03 \\pm 0{,}02)$ cm. ¿Tiene sentido promediarlas? ¿Cómo las combinarías correctamente?(Pista: promedio ponderado por $1/\\sigma^2$ — lo vamos a formalizar en el Colab 06.)